# Exploratory Data Analysis

In [ ]:
# Import Python libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import seaborn as sns

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('../data/raw/flash_floods_ky_2015_2025.csv')

In [ ]:
# Read first rows of dataset
df.head()

In [ ]:
# Read last rows of dataset
df.tail()

In [ ]:
# Check data type and missing values
df.info()

print(df.columns.tolist())

The EVENT_NARRATIVE column has 1749 non-null values. 

In [ ]:
# See the dimensions of the dataset
df.shape

In [ ]:
# Find categorical variables using .select_dtypes()
df.select_dtypes(include='object').columns

Columns such as BEGIN_DATE and END_DATE may need their data types changed since I plan to conduct numerical analysis with them. 

In [ ]:
# Explore value counts for various categorial variables
df.value_counts(subset=['FLOOD_CAUSE']),
df.value_counts(subset=['CZ_NAME_STR'])

print(df['FLOOD_CAUSE'].value_counts())
print(df['CZ_NAME_STR'].value_counts())

There is an outlier in the FLOOD_CAUSE column (Heavy Rain / Tropical System). 

In [ ]:
# Find row that has the Heavy Rain / Tropical System flood cause
heavy_rain_row = df.loc[df['FLOOD_CAUSE'] == 'Heavy Rain / Tropical System'].iloc[0]
print(heavy_rain_row)

In [ ]:
# Sort values using .sort_values()
df.sort_values(by='BEGIN_DATE', ascending=True)

In [ ]:
# Rename columns using .rename()
df.rename(columns={'CZ_NAME_STR': 'COUNTY_NAME'}, inplace=True)

print(df.columns.tolist())

I should drop the MAGNITUDE, MAGNITUDE_TYPE, TOR_F_SCALE, TOR_LENGTH, and TOR_WIDTH columns since there aren't any values within those columns. Plus, these columns aren't relevant to the analysis that I'm conducting at this time.  

In [ ]:
# Drop columns using .drop()
df.drop(columns=['MAGNITUDE', 'MAGNITUDE_TYPE', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH'], inplace=True)

print(df.columns.tolist())

In [ ]:
# Fix data types using .astype()
df = df.astype({'BEGIN_DATE': 'datetime64[ns]'})
df = df.astype({'END_DATE': 'datetime64[ns]'})
df = df.astype({'EVENT_ID': 'string'})
df = df.astype({'BEGIN_TIME': 'string'})
df = df.astype({'END_TIME': 'string'})
df = df.astype({'EPISODE_ID': 'string'})
df = df.astype({'CZ_FIPS': 'string'})
df = df.astype({'EVENT_NARRATIVE': 'string'})
df = df.astype({'EPISODE_NARRATIVE': 'string'})

print(df.dtypes)

I decided to change the data type of columns such as EVENT_ID, EVENT_NARRATIVE, and EPISODE_NARRATIVE to string since I will not be using these columns for numerical analysis at this time. 

In [ ]:
# Identify missing values using .isnull() and/or .notnull()
df.isnull().sum()

In [ ]:
# View rows with missing values
df[df.isnull().any(axis=1)] 

print(df[df['EVENT_NARRATIVE'].isnull()])

EVENT_NARRATIVE values are missing from 07-19-2023. I'm not going to be using EVENT_NARRATIVE data at this time, so I will just fill those missing values in with "Unknown." 

In [ ]:
# Replace missing values in EVENT_NARRATIVE with "Unknown"
df["EVENT_NARRATIVE"] = df["EVENT_NARRATIVE"].fillna("Unknown")

print(df["EVENT_NARRATIVE"].isna().sum())

In [ ]:
# Check for duplicate rows using .duplicated()
df.duplicated().sum()

In [ ]:
# Create YEAR column
df["YEAR"] = df["BEGIN_DATE"].dt.year

df = df.astype({'YEAR': 'int64'})

print(df["YEAR"].value_counts())

In [ ]:
# Check data to ensure that YEAR column was added correctly. 
df.info()

In [ ]:
# Save cleaned dataset to new csv file
df.to_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv', index=False)

In [ ]:
# Use .groupby() to explore number of flash flooding events by county.
county_counts = (
    df.groupby("COUNTY_NAME")
      .size()
      .reset_index(name="Total_Events")
      .sort_values("Total_Events", ascending=False)
)

# View top 10 counties with most flash flooding events
print(county_counts.head(10))

In [ ]:
# Use .groupby() to explore total property damage by county
property_damage = (
    df.groupby("COUNTY_NAME")["DAMAGE_PROPERTY_NUM"]
      .sum()
      .reset_index()
      .sort_values("DAMAGE_PROPERTY_NUM", ascending=False)
)

# View top 10 counties with most property damage from flash flooding events
print(property_damage.head(10))

In [ ]:
# Use .groupby() to explore total deaths and injuries by county
human_impact = (
    df.groupby("COUNTY_NAME")[["DEATHS_DIRECT", "INJURIES_DIRECT"]]
      .sum()
      .reset_index()
      .sort_values("DEATHS_DIRECT", ascending=False)
)

# View top 10 counties with the most direct deaths and injuries from flash flooding events
print(human_impact.head(10))

In [ ]:
# Use .groupby() to explore how many flash flooding events were reported from each source
source_counts = (
    df.groupby("SOURCE")
      .size()
      .reset_index(name="Total_Events")
      .sort_values("Total_Events", ascending=False)
)

print(source_counts)

911 Call Centers have reported the most flash flooding events from 2015 - 2025.

In [ ]:
# Use .groupby() to explore the top 5 dates that had the most flash flooding events
date_counts = (
    df.groupby("BEGIN_DATE")
      .size()
      .reset_index(name="TOTAL_EVENTS")
      .sort_values("TOTAL_EVENTS", ascending=False)
)

print(date_counts.head(5))

In [ ]:
# Use .groupby() to explore the counties and cities that experienced flash flooding events on the top 5 dates. Show the dates and the total number of events for each county on those dates.
top_dates = date_counts.head(5)["BEGIN_DATE"]
county_city_date_counts = (
    df[df["BEGIN_DATE"].isin(top_dates)]
      .groupby(["BEGIN_DATE", "COUNTY_NAME", "BEGIN_LOCATION"])
      .size()
      .reset_index(name="TOTAL_EVENTS")
      .sort_values(["BEGIN_DATE", "TOTAL_EVENTS"], ascending=[True, False])
)

print(county_city_date_counts)

In [ ]:
# Import County/City counts data into csv file
county_city_date_counts.to_csv("../data/processed/county_city_date_counts.csv", index=False)

In [ ]:
# Sort values to determine which flash flooding events were the most expensive
highest_damage = df.sort_values("DAMAGE_PROPERTY_NUM", ascending=False)

print(highest_damage[
    ["BEGIN_DATE", "COUNTY_NAME", "DAMAGE_PROPERTY_NUM"]
].head(10))

In [ ]:
# Sort Values to determine which flash flooding events had the most injuries
most_injuries = df.sort_values("INJURIES_DIRECT", ascending=False)

print(most_injuries[
    ["BEGIN_DATE", "COUNTY_NAME", "INJURIES_DIRECT"]
].head(10))

In [ ]:
# Create a summary table for the following question: What has been the total statewide impact of flash flooding in Kentucky from 2015-2025?
summary = pd.DataFrame({
    "Total Events": [len(df)],
    "Total Property Damage": [df["DAMAGE_PROPERTY_NUM"].sum()],
    "Total Deaths": [df["DEATHS_DIRECT"].sum()],
    "Total Injuries": [df["INJURIES_DIRECT"].sum()]
})

print(summary)

In [ ]:
# Create a summary table for the following question: How has reporting sources changed over time?
source_year = pd.crosstab(df["YEAR"], df["SOURCE"])

print(source_year)

This information can be used to create a heatmap of how reporting has changed over time for each source. 

In [ ]:
# Create a heatmap that shows correlations between numerical variables
cols = [
    "YEAR",
    "DEATHS_DIRECT",
    "INJURIES_DIRECT",
    "DAMAGE_PROPERTY_NUM",
    "DAMAGE_CROPS_NUM"
]

corr_matrix = df[cols].corr()

im = plt.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(im, label="Pearson Correlation")

plt.title("No Strong Correlations Between Year and Flash Flooding Impact Variables", fontsize=14, pad=20, ha="center")

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha="right")
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)

plt.show()

1. What question or idea does this chart help answer?
This chart shows whether there are linear relationships between year and key flash flood impact variables, as well as relationships among the impact variables themselves. It helps identify whether any variables tend to increase, decrease, or move together over time.
2. Why is this chart type appropriate?
A correlation heatmap is appropriate because it allows multiple relationships to be visualized in a single view. It summarizes how several numeric variables relate to one another without needing many separate plots.
3. What design choices did you make (color, labels, order, scale)?
I used a diverging “coolwarm” color palette with a fixed range from -1 to 1 to clearly show direction and strength of correlations. I rotated labels for readability. I kept axis names consistent to make comparisons easier.
4. How did you ensure accuracy and avoid misleading design?
I ensured that the heatmap used Pearson correlation values directly from the dataset without transformation, ensuring mathematical accuracy. The full -1 to 1 scale and consistent labeling aids in preventing exaggeration or misinterpretation of relationships.

In [ ]:
# Create a heatmap that shows correlations between years and sources
source_year = pd.crosstab(df["YEAR"], df["SOURCE"])

plt.figure(figsize=(14, 10))

ax = sns.heatmap(
    source_year,
    annot=True,
    cmap="YlGnBu",
    fmt="d"
)

cbar = ax.collections[0].colorbar
cbar.set_label("Number of Flash Flood Events", labelpad=10)

plt.title(
    "911 Call Centers Have Reported the Most Flash Flooding Events from 2015-2025",
    fontsize=14,
    fontweight="bold",
    pad=15
)

plt.xlabel(
    "Source",
    fontsize=12,
    labelpad=10
)

plt.ylabel(
    "Year",
    fontsize=12,
    labelpad=10
)

ax.set_xticklabels(
    source_year.columns,
    rotation=65,
    ha="right"
)

ax.set_yticklabels(
    source_year.index,
    rotation=0,
    ha="right"
)

offset = mtransforms.ScaledTranslation(
    8 / 78,   
    0,
    plt.gcf().dpi_scale_trans
)

for label in ax.get_xticklabels():
    label.set_transform(label.get_transform() + offset)

plt.tight_layout()
plt.show()

1. What question or idea does this chart help answer?
This chart shows how the number of flash flood events varies by reporting source across each year. It helps identify which sources, such as 911 call centers, report the most events over time.
2. Why is this chart type appropriate?
A heatmap is appropriate because it allows comparison of counts across two categorical variables (year and source) in a compact visual format. It makes patterns in reporting volume easy to spot across both dimensions.
3. What design choices did you make (color, labels, order, scale)?
I chose to use the “YlGnBu” color palette to represent increasing event counts, with annotations added to show exact values in each cell. I made adjustments to axis labels, rotations, and spacing adjustments to improve readability of densely packed categories.
4. How did you ensure accuracy and avoid misleading design?
I ensured that the chart used raw counts from a crosstab without normalization so that values directly reflected event totals. I used Consistent labeling, clear axis definitions, and annotated values to help prevent misinterpretation of relative differences.

In [ ]:
# Create a line graph that shows total flash flooding events per year from 2015-2025
events_per_year = df["YEAR"].value_counts().sort_index()

plt.figure(figsize=(12, 6))

ax = plt.gca()

ax.plot(
    events_per_year.index,
    events_per_year.values,
    marker="o",
    linestyle="--",
    linewidth=1,
    color="#6272dc"
)

ax.set_title(
    "Has the Frequency of Flash Flooding Events\nIncreased in Kentucky from 2015–2025?",
    fontsize=16,
    fontweight="bold",
    pad=18
)

ax.set_xlabel(
    "Year",
    fontsize=12,
    labelpad=10
)

ax.set_ylabel(
    "Total Events",
    fontsize=12,
    labelpad=10
)

ax.set_xticks(events_per_year.index)
ax.set_xticklabels(
    events_per_year.index,
    rotation=35,
    ha="center"
)

ax.set_ylim(0, events_per_year.max() + 5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

1. What question or idea does this chart help answer?
This chart examines whether the frequency of flash flooding events in Kentucky has increased over time from 2015 to 2025. It highlights year-to-year trends in total event counts to show overall change patterns.
2. Why is this chart type appropriate?
A line chart is appropriate because it effectively shows trends over time and makes it easy to see increases or decreases across consecutive years. It is well-suited for time-series data like annual event counts.
3. What design choices did you make (color, labels, order, scale)?
I decided to use single blue line with markers and a dashed style to clearly show both trend and individual data points, while axis labels and rotated year ticks were changed to improve readability. I adjusted the y-axis to start at zero to avoid exaggerating changes.
4. How did you ensure accuracy and avoid misleading design?
I used sorted yearly counts to maintain correct chronological order and included a consistent y-axis scale starting at zero. I also removed unnecessary spines and used clear labeling to prevent visual distortion or misinterpretation of trends.

In [ ]:
# Create a bar chart that shows the top 10 counties with the most flash flooding events
county_counts = (
    df.groupby("COUNTY_NAME")
      .size()
      .reset_index(name="Total_Events")
      .sort_values("Total_Events", ascending=False)
      .head(10)
)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=county_counts,
    x="COUNTY_NAME",
    y="Total_Events",
    color="steelblue"
)

plt.title(
    "Which County Has Experienced the Most Flash Flooding Events from 2015-2025?",
    fontsize=16,
    fontweight="bold",
    pad=10
)

plt.xlabel(
    "County",
    fontsize=12,
    labelpad=8
)

plt.ylabel(
    "Total Events",
    fontsize=12,
    labelpad=8
)

plt.xticks(
    rotation=35,
    ha="center",
    fontsize=11
)

plt.yticks(
    va="center",
    fontsize=11
)

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

1. What question or idea does this chart help answer?
This chart identifies which top 10 counties in Kentucky experienced the highest number of flash flooding events from 2015 to 2025. It helps compare event frequency across these top 10 counties.
2. Why is this chart type appropriate?
A bar chart is appropriate because it clearly compares discrete categories (counties) based on their total event counts. It makes it easy to rank and visually distinguish differences between counties.
3. What design choices did you make (color, labels, order, scale)?
I decided to use a single “steelblue” color for the bars to keep focus on comparison rather than aesthetics. I sorted counties in descending order to highlight the highest-impact locations. I adjusted axis labels, rotation, and font sizing to improve readability.
4. How did you ensure accuracy and avoid misleading design?
I grouped and sorted directly from raw event counts without transformation, ensuring accurate totals for each county. I limited the plot to the top 10 counties in order to prevent clutter while still accurately representing the highest values.